In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv('balanced_dataset_12k.csv')

In [3]:
df.head()

,headline,label,confidence
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,0.65
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,0.85
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,0.75
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,0.45
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,0.45


In [4]:
df = df.drop(columns=['confidence'], axis = 1)

In [5]:
df['clean_text'] = df['headline'].str.lower()
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"how jd(s) is quietly remaking its leadership team: as a veteran mla is removed from core committee, ‘first family’ scions get key posts"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"waqf bill: more than 12 hours of debate, and a washroom break that got congress all wet"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"india-u.s. interim trade deal: ‘namaste trump scored over howdy modi’, says congress"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"‘richest’ cm chandrababu naidu’s draws almost entirely on one asset, mamata records sharpest decline"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’: how ex-congress leader struggled to find feet in bjp


In [6]:
import string
def remove_punctuations(text):
    punctuations = string.punctuation
    return text.translate(str.maketrans('', '', punctuations))

df['clean_text'] = df['clean_text'].apply(lambda x: remove_punctuations(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,how jds is quietly remaking its leadership team as a veteran mla is removed from core committee ‘first family’ scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill more than 12 hours of debate and a washroom break that got congress all wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal ‘namaste trump scored over howdy modi’ says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,‘richest’ cm chandrababu naidu’s draws almost entirely on one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’ how excongress leader struggled to find feet in bjp


In [7]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

"a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't, as, at, be, because, been, before, being, below, between, both, but, by, can, couldn, couldn't, d, did, didn, didn't, do, does, doesn, doesn't, doing, don, don't, down, during, each, few, for, from, further, had, hadn, hadn't, has, hasn, hasn't, have, haven, haven't, having, he, he'd, he'll, her, here, hers, herself, he's, him, himself, his, how, i, i'd, if, i'll, i'm, in, into, is, isn, isn't, it, it'd, it'll, it's, its, itself, i've, just, ll, m, ma, me, mightn, mightn't, more, most, mustn, mustn't, my, myself, needn, needn't, no, nor, not, now, o, of, off, on, once, only, or, other, our, ours, ourselves, out, over, own, re, s, same, shan, shan't, she, she'd, she'll, she's, should, shouldn, shouldn't, should've, so, some, such, t, than, that, that'll, the, their, theirs, them, themselves, then, there, these, they, they'd, they'll, they're, they've, this, those, through, to, too, under, until, up, 

In [9]:
STOPWORDS = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in STOPWORDS])

df['clean_text'] = df['clean_text'].apply(lambda x: remove_stopwords(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,jds quietly remaking leadership team veteran mla removed core committee ‘first family’ scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal ‘namaste trump scored howdy modi’ says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,‘richest’ cm chandrababu naidu’s draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’ excongress leader struggled find feet bjp


In [10]:
import re

def clean_text(text):
    # 1. Normalize weird unicode quotes → standard ones
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # 2. Replace ellipsis with normal dots
    text = text.replace("…", "...")

    # 3. Remove unwanted symbols (but KEEP ! ? ' ")
    text = re.sub(r"[^a-zA-Z0-9\s!?\'\"]", " ", text)

    # 4. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df['clean_text'] = df['clean_text'].apply(lambda x: clean_text(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,jds quietly remaking leadership team veteran mla removed core committee 'first family' scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal 'namaste trump scored howdy modi' says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,'richest' cm chandrababu naidu's draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar 'resignation' excongress leader struggled find feet bjp


In [11]:
from nltk.tokenize import TweetTokenizer

tokenizer = TweetTokenizer()
df['clean_text'] = df['clean_text'].apply(lambda x: tokenizer.tokenize(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"[jds, quietly, remaking, leadership, team, veteran, mla, removed, core, committee, ', first, family, ', scions, get, key, posts]"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"[waqf, bill, 12, hours, debate, washroom, break, got, congress, wet]"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"[indiaus, interim, trade, deal, ', namaste, trump, scored, howdy, modi, ', says, congress]"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"[', richest, ', cm, chandrababu, naidu's, draws, almost, entirely, one, asset, mamata, records, sharpest, decline]"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,"[sunil, jakhar, ', resignation, ', excongress, leader, struggled, find, feet, bjp]"


In [12]:
df['text_for_bow'] = df['clean_text'].apply(lambda x: " ".join(x))

In [13]:
df.head()

,headline,label,clean_text,text_for_bow
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"[jds, quietly, remaking, leadership, team, veteran, mla, removed, core, committee, ', first, family, ', scions, get, key, posts]",jds quietly remaking leadership team veteran mla removed core committee ' first family ' scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"[waqf, bill, 12, hours, debate, washroom, break, got, congress, wet]",waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"[indiaus, interim, trade, deal, ', namaste, trump, scored, howdy, modi, ', says, congress]",indiaus interim trade deal ' namaste trump scored howdy modi ' says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"[', richest, ', cm, chandrababu, naidu's, draws, almost, entirely, one, asset, mamata, records, sharpest, decline]",' richest ' cm chandrababu naidu's draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,"[sunil, jakhar, ', resignation, ', excongress, leader, struggled, find, feet, bjp]",sunil jakhar ' resignation ' excongress leader struggled find feet bjp


In [14]:
df['text_for_bow'] = df['clean_text'].apply(
    lambda tokens: " ".join([word for word in tokens if word != "'"])
)

In [15]:
df.head()

,headline,label,clean_text,text_for_bow
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"[jds, quietly, remaking, leadership, team, veteran, mla, removed, core, committee, ', first, family, ', scions, get, key, posts]",jds quietly remaking leadership team veteran mla removed core committee first family scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"[waqf, bill, 12, hours, debate, washroom, break, got, congress, wet]",waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"[indiaus, interim, trade, deal, ', namaste, trump, scored, howdy, modi, ', says, congress]",indiaus interim trade deal namaste trump scored howdy modi says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"[', richest, ', cm, chandrababu, naidu's, draws, almost, entirely, one, asset, mamata, records, sharpest, decline]",richest cm chandrababu naidu's draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,"[sunil, jakhar, ', resignation, ', excongress, leader, struggled, find, feet, bjp]",sunil jakhar resignation excongress leader struggled find feet bjp


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

tfidf = TfidfVectorizer(
    max_features=5000,        # limit vocab size
    ngram_range=(1,2),        # unigrams + bigrams
    min_df=2,                 # ignore rare words
    max_df=0.9,               # ignore very common words
    sublinear_tf=True         # improves performance
)

X = tfidf.fit_transform(df['text_for_bow'])

y = df['label']

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=-1,              # use all cores (recommended)
    random_state=42,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None,
    monotonic_cst=None
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
y_test_pred = rf.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score, classification_report

print("Validation Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Validation Accuracy: 0.6571913449299958
               precision    recall  f1-score   support

non-sarcastic       0.69      0.57      0.62      1179
    sarcastic       0.63      0.75      0.69      1178

     accuracy                           0.66      2357
    macro avg       0.66      0.66      0.65      2357
 weighted avg       0.66      0.66      0.65      2357



In [21]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    C=1.0,
    tol=0.001,
    class_weight=None,
    max_iter=10000,   # IMPORTANT: higher for convergence
    random_state=42
)

svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [22]:
y_val_pred = svm.predict(X_test)

In [23]:
print("Validation Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Validation Accuracy: 0.6571913449299958
               precision    recall  f1-score   support

non-sarcastic       0.69      0.57      0.62      1179
    sarcastic       0.63      0.75      0.69      1178

     accuracy                           0.66      2357
    macro avg       0.66      0.66      0.65      2357
 weighted avg       0.66      0.66      0.65      2357



In [24]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300, 500],

    'max_depth': [None, 10, 20, 30, 50],

    'min_samples_split': [2, 5, 10],

    'min_samples_leaf': [1, 2, 4],

    'max_features': ['sqrt', 'log2', None],

    'bootstrap': [True, False],

    'criterion': ['gini', 'entropy']
}

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    # CORE (most impact)
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 30, None],
    'max_features': ['sqrt', 0.5],

    # REGULARIZATION (important but controlled)
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 3],

    # LIGHT extras (small exploration only)
    'bootstrap': [True],
    'criterion': ['gini', 'log_loss']
}

halving = HalvingGridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid,
    cv=cv,
    factor=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=3
)

halving.fit(X_train, y_train)

print("Best Params:", halving.best_params_)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 116
max_resources_: 9427
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 144
n_resources: 116
Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV 5/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=0.750, test=0.515) total time=   0.2s
[CV 3/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=(train=0.841, test=0.309) total time=   0.6s
[CV 4/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200;, score=(train=0.854, test=0.471) total time=   0.4s
[CV 1/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=3, min_samples_split=2, n_estimators=100;, score=(train=0.358, test=0.358

In [25]:
best_rf = halving.best_estimator_

y_test_pred = best_rf.predict(X_test)

In [26]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.6453118370810352
               precision    recall  f1-score   support

non-sarcastic       0.66      0.59      0.63      1179
    sarcastic       0.63      0.70      0.66      1178

     accuracy                           0.65      2357
    macro avg       0.65      0.65      0.64      2357
 weighted avg       0.65      0.65      0.64      2357



In [27]:
from sklearn.svm import LinearSVC

param_grid = {
    'C': np.logspace(-3, 2, 8),
    'class_weight': [None, 'balanced'],
    'tol': [1e-3, 1e-4, 1e-5],
    'max_iter': [5000, 10000]   # removed 1000 (too small)
}

halving1 = HalvingGridSearchCV(
    LinearSVC(),   # ✅ FIXED
    param_grid,
    cv=cv,
    factor=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

halving1.fit(X_train, y_train)

print("Best Params:", halving1.best_params_)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 116
max_resources_: 9427
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 96
n_resources: 116
Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 2/5] END bootstrap=True, criterion=log_loss, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=1.000, test=0.635) total time=   0.5s
[CV 1/5] END bootstrap=True, criterion=log_loss, max_depth=30, max_features=0.5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=0.960, test=0.432) total time=   1.2s
[CV 5/5] END bootstrap=True, criterion=log_loss, max_depth=30, max_features=0.5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=(train=0.986, test=0.548) total time=   1.2s
[CV 3/5] END bootstrap=True, criterion=log_loss, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=(train=1.

In [30]:
best_svm = halving1.best_estimator_

y_test_pred = best_svm.predict(X_test)

In [31]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.6758591429783624
               precision    recall  f1-score   support

non-sarcastic       0.68      0.67      0.67      1179
    sarcastic       0.67      0.68      0.68      1178

     accuracy                           0.68      2357
    macro avg       0.68      0.68      0.68      2357
 weighted avg       0.68      0.68      0.68      2357



In [32]:
import joblib

joblib.dump(best_rf, "rf_tfidf_model.joblib")
joblib.dump(best_svm, "lsvm_ tfidf_model.joblib")

['lsvm_ tfidf_model.joblib']